In [54]:
# ============================================
# Ollama + Qwen3-8B-Q4_K_M.gguf RAG 完全版
# ============================================

import os
import psycopg2
from pgvector.psycopg2 import register_vector
import fitz  # PyMuPDF
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
import ollama


# ============================================
# 3. PDF → テキスト
# ============================================
def load_pdf_text(path: str) -> str:
    doc = fitz.open(path)
    texts = []
    for page in doc:
        texts.append(page.get_text("text"))
    return "\n".join(texts)


# ============================================
# 3-2. TXT → テキスト
# ============================================
def load_txt_text(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


# ============================================
# 3-3. PDF/TXT 自動判定
# ============================================
def load_document_text(path: str) -> str:
    if path.lower().endswith(".pdf"):
        return load_pdf_text(path)
    elif path.lower().endswith(".txt"):
        return load_txt_text(path)
    else:
        return ""


# ============================================
# 8. 追加 DOC（PDF/TXT）追記
# ============================================
def load_txt(doc_dir: str):
    results = []

    for filename in os.listdir(doc_dir):
        if not (filename.lower().endswith(".pdf") or filename.lower().endswith(".txt")):
            continue

        path = os.path.join(doc_dir, filename)
        print(f"[追加DOC] {path}")

        text = load_document_text(path)

        # スコアは仮で 1.0（後で FAISS や pgvector に置き換え可能）
        results.append((filename, text, 1.0))

    return results

def trim_to_token_limit(text: str, max_chars: int = 5500) -> str:
    """
    Ollama の 4096 tokens に安全に収まるように
    日本語テキストを最大 max_chars 文字に切り詰める。
    """
    if len(text) <= max_chars:
        return text
    return text[:max_chars]

# ============================================
# 10. Ollama LLM（Qwen3-8B-Q4_K_M.gguf）
# ============================================
def generate_answer(prompt: str) -> str:
    out = ollama.chat(
        model="Qwen3-8B-Q4_K_M",   # ★ここを固定
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return out["message"]["content"].strip()


# ============================================
# 11. プロンプト構築
# ============================================
def build_prompt(contexts):
    ctx_blocks = []

    for source, text, score in contexts:
        # 4096 tokens に収まるように本文を切り詰める
        trimmed = trim_to_token_limit(text, max_chars=5500)
        ctx_blocks.append(f"[source={source}, score={score:.4f}]\n{trimmed}")

    input_text = "\n\n".join(ctx_blocks)

    prompt = f"""
以下のテキストから、本文とは関係のない情報をすべて削除してください。

削除対象の例：
- Edge ブラウザのタブ情報（pageTitle, pageUrl, tabId, isCurrent など）
- 「User's Edge browser tabs metadata」などのシステム説明文
- URL や検索クエリ
- Copilot やブラウザの動作説明
- JSON 形式のメタデータ
- コードに混入したブラウザ情報
- 本文と無関係な注釈・説明・ログ

残すべきもの：
- 本文として意味を持つ文章のみ
- 要約しない
- PDF/TXT から抽出された本来の内容

出力は「該当の文章をそのまま」出力してください。
余計な説明やコメントは不要です。

=== テキスト開始 ===
{input_text}
=== テキスト終了 ===
"""

    return prompt


In [55]:

# ============================================
# 14. main
# ============================================
if __name__ == "__main__":
    PDF_DIR = r"C:\Users\USER\Documents\RAG_system\rag_chunking_withLLM\docs"

    print("\n=== 実行 ===")
    contexts = load_txt(PDF_DIR)
    prompt = build_prompt(contexts)
    answer = generate_answer(prompt)
    print(answer)


=== 実行 ===
[追加DOC] C:\Users\USER\Documents\RAG_system\rag_chunking_withLLM\docs\minkabu.jp_news_4080030.txt
以下は、不要な情報（ブラウザタブ情報、URL、システム説明など）を除去した、**ネクセラファーマのビジネスモデルとビジョンに関する文章**です。

---

**バイオ企業が薬の7割を発明している**  
販売している医薬品会社と開発元が違うことは、スライドのデータからも見てとれます。スライド左側の棒グラフは全販売高のうち誰が開発したかを示しており、折れ線グラフは全販売高のうち大手製薬以外が販売している割合を示しています。  
大手と言えば、日本で言えばみなさまがご存知の武田薬品やアステラス製薬などを指します。折れ線グラフを見ると、2023年時点で、大手以外が販売している医薬品は、全体の約23パーセントを占めています。  
ただし、棒グラフを見ていただくとわかるとおり、現在販売されている医薬品売り上げの大部分が、大手以外が創出したものになります。したがって、大手以外がいかに研究・開発を担うか、というのが今後の医薬品市場において大事なポイントとなってきます。

**バイオ企業とバイオ医薬品企業のギャップ**  
大手以外の企業のうち、いわゆるスタートアップなどはバイオ企業と呼ばれています。バイオ企業と、大手製薬すなわちバイオ医薬品企業は、薬を作って販売するという点では同じですが、ビジネスモデルにおいて大きなギャップがあると言われています。  
バイオ企業は売上がない、もしくは大きく変動します。製品・開発品が1つだけある「一本足打法」と言われる状態が多く、開発のマイルストンなどで獲得するような売上は年によって変わってしまいます。すると、ある年では売上がない、ある年は大きな売上があるということが起きます。  
そのような売上の特徴があるため、利益が少ない、もしくは大赤字が、バイオ企業としては一般的だと言われています。  
また、バイオ企業に関しては、パイプラインの価値のみで評価されているとも言えます。製品が当たれば大きな利益が入ってくるのですが、実際に成功するかどうかは開発してみないとわかりません。ただし、パイプラインの価値が大きいため、このくらいの時価総額